# Regressão: reta, polinômios e regularização

**Para quem está começando.** Nenhuma linha de código aqui assume que você já viu regressão. A ideia é rodar de cima para baixo, célula por célula, e olhar os números.

O que vamos fazer, em ordem:

1. ajustar uma reta e ver o que ela não captura;
2. aumentar o grau do polinômio até o modelo começar a decorar o treino;
3. penalizar os coeficientes com Ridge e Lasso;
4. comparar com vizinhos, árvore e SVM na mesma divisão.

Roda no Google Colab ou no Jupyter do seu computador. Só precisa de `numpy`, `pandas`, `scikit-learn` e `matplotlib`, que já vêm no Colab.


## O que muda em relação à classificação

O alvo agora é um número, não uma classe. Então as métricas mudam: em vez de acurácia, medimos erro. O **RMSE** e o **MAE** são erros na unidade do alvo (quanto menor, melhor) e o **R²** é a fração da variação explicada (quanto maior, melhor). RMSE pune erro grande mais que MAE.

O problema central também muda de nome, mas não de natureza: em vez de acertar a fronteira entre duas classes, o modelo precisa generalizar para linhas que não viu. A pergunta que organiza este notebook é quanto de capacidade o modelo deve ter:

- pouca capacidade, e ele erra nos dois conjuntos (underfitting);
- capacidade demais, e ele acerta no treino e erra no teste (overfitting).

Regularização é a terceira via, e é o assunto da segunda metade do notebook.


## Passo 0: ambiente e configuração

O conjunto é o de preços medianos de casas na Califórnia, o mesmo usado nos outros notebooks deste repositório: 20 640 linhas, das quais sorteamos 2 500 para o notebook rodar rápido. O recorte é o mesmo em todas as células.


In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Lasso, LassoCV, LinearRegression, Ridge, RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42
TARGET = "target"
sns.set_theme(context="notebook", style="whitegrid", palette="colorblind",
              rc={"figure.figsize": (9, 4)})

housing = fetch_california_housing()
frame = pd.DataFrame(housing.data, columns=housing.feature_names)
frame[TARGET] = housing.target
sel = frame.sample(2500, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    sel.drop(columns=TARGET), sel[TARGET], test_size=0.25, random_state=RANDOM_STATE,
)
print(f"✅ {X_train.shape[0]} linhas de treino | {X_test.shape[0]} de teste "
      f"| {X_train.shape[1]} atributos | alvo medio {y_train.mean():.3f}")

## Passo 1: uma reta e polinômios de grau crescente

A regressão linear com 8 atributos não desenha uma reta, desenha um plano em 8 dimensões: ela soma cada atributo multiplicado por um coeficiente. Se o efeito de um atributo não for proporcional, ela erra.

O jeito de dar capacidade ao modelo sem trocar de algoritmo é criar colunas novas a partir das existentes: o quadrado de cada uma e o produto de cada par. É o que o `PolynomialFeatures` faz, e o número de colunas cresce rápido. É por isso que o grau é um botão perigoso.


In [ ]:
def rmse(modelo, X, y) -> float:
    """Root mean squared error of a fitted model on a given table."""
    return root_mean_squared_error(y, modelo.predict(X))


linhas = []
modelos_por_grau = {}
for grau in range(1, 9):
    pipeline = make_pipeline(PolynomialFeatures(grau, include_bias=False), StandardScaler(),
                             LinearRegression())
    pipeline.fit(X_train, y_train)
    modelos_por_grau[grau] = pipeline
    linhas.append({"grau": grau, "colunas": pipeline[:-1].transform(X_train[:1]).shape[1],
                   "rmse_treino": rmse(pipeline, X_train, y_train),
                   "rmse_validacao": rmse(pipeline, X_test, y_test)})

tabela_graus = pd.DataFrame(linhas).set_index("grau")
display(tabela_graus.round(4))

In [ ]:
fig, ax = plt.subplots()
ax.plot(tabela_graus.index, tabela_graus["rmse_treino"], marker="o", label="treino", color="#0F6B68")
ax.plot(tabela_graus.index, tabela_graus["rmse_validacao"], marker="o", label="validação",
        color="#B45309")
ax.set_yscale("log")
ax.set_xlabel("grau do polinômio")
ax.set_ylabel("RMSE (escala log)")
ax.set_title("Mais capacidade nem sempre é melhor")
ax.legend()
plt.show()

### Como ler isso

A curva de validação tem forma de U, e as duas metades dizem coisas diferentes:

- à esquerda, treino e validação erram juntos. O modelo é simples demais para o dado (underfitting);
- à direita, o erro de treino continua caindo e o de validação sobe. O modelo passou a decorar: ele descreve o ruído daquele treino, que não se repete no teste (overfitting).

O grau escolhido é o do mínimo da curva de validação, não o de menor erro de treino. E repare na coluna `colunas`: o grau 5 passa de 1 200 colunas e o grau 8 passa de 12 000, para 1 875 linhas de treino. É esse desequilíbrio que o polinômio cria, e é ele que o mínimo quadrado usa para decorar.

Um cuidado: esse mínimo é o mínimo **desta** divisão. Com outra semente ele se desloca, e a diferença entre graus vizinhos costuma ser menor que o ruído da partição.


## Passo 2: penalizar os coeficientes

Regularização é somar à função de perda uma penalidade pelo tamanho dos coeficientes. O efeito prático: o modelo só paga o preço de usar uma coluna se ela compensar. Há duas versões, e a diferença aparece na saída:

- **Ridge** (penalidade L2) encolhe todos os coeficientes na direção do zero, sem zerar nenhum;
- **Lasso** (penalidade L1) pode zerar coeficientes, e aí ele vira também um seletor de atributos.

A força da penalidade é o `alpha`, escolhido por validação cruzada. E a padronização deixa de ser opcional: sem ela a penalidade depende da unidade de cada coluna, e a coluna medida em milhares é penalizada muito mais que a medida em décimos.


In [ ]:
alphas = np.logspace(-3, 3, 40)

linhas = []
for nome, modelo in {"Ridge": RidgeCV(alphas=alphas), "Lasso": LassoCV(alphas=alphas, max_iter=20000)}.items():
    pipeline = make_pipeline(StandardScaler(), modelo).fit(X_train, y_train)
    coeficientes = pipeline[-1].coef_
    linhas.append({
        "modelo": nome,
        "alpha": pipeline[-1].alpha_,
        "coeficientes zerados": int(np.sum(np.isclose(coeficientes, 0.0))),
        "rmse_validacao": rmse(pipeline, X_test, y_test),
    })

tabela_penalizada = pd.DataFrame(linhas).set_index("modelo")
display(tabela_penalizada.round(4))

sem_penalidade = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
print(f"📊 sem penalidade: RMSE de validação = {rmse(sem_penalidade, X_test, y_test):.4f}")

In [ ]:
padronizado = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas)).fit(X_train, y_train)
lasso = make_pipeline(StandardScaler(), LassoCV(alphas=alphas, max_iter=20000)).fit(X_train, y_train)

ordem = np.argsort(np.abs(padronizado[-1].coef_))
posicoes = np.arange(len(ordem))
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(posicoes - 0.25, padronizado[-1].coef_[ordem], height=0.25, label="sem penalidade",
        color="#0F6B68")
ax.barh(posicoes, ridge[-1].coef_[ordem], height=0.25, label="Ridge", color="#B45309")
ax.barh(posicoes + 0.25, lasso[-1].coef_[ordem], height=0.25, label="Lasso", color="#6B7280")
ax.set_yticks(posicoes)
ax.set_yticklabels(X_train.columns[ordem], fontsize=8)
ax.set_xlabel("coeficiente (atributos padronizados)")
ax.set_title("Ridge encolhe, Lasso zera")
ax.legend()
plt.show()

### Como ler isso

As três barras dizem o que cada método faz com o mesmo dado: sem penalidade, os coeficientes ficam do tamanho que o mínimo quadrado pedir; o Ridge encolhe todos; o Lasso zera os que não se pagam, e a coluna `coeficientes zerados` conta quantos.

Aqui o RMSE de validação ficou um pouco **pior** que o do modelo sem penalidade, e isso é o esperado neste caso: são 8 colunas e nenhuma delas é inútil, então a penalidade só cobra o seu preço. O que ela compra é estabilidade, e o ganho aparece quando há muitas colunas correlacionadas ou poucas linhas, que é onde o mínimo quadrado fica instável. O Lasso ainda zerou alguns coeficientes, o que também custou acurácia aqui: selecionar atributo é útil quando existe coluna de enfeite, e prejudicial quando não existe.

Duas armadilhas: sem padronizar, o `alpha` escolhido não significa nada (a penalidade é aplicada em unidades diferentes); e comparar `alpha` entre dois conjuntos de dados diferentes tampouco significa nada, porque ele depende da escala do alvo e das colunas.


## Passo 3: os outros modelos, na mesma divisão

Regressão não é só a família linear. A mesma tabela de comparação do notebook de classificação, agora com RMSE, MAE e R², mais o tempo de treino e de predição.


In [ ]:
def timeit(fn, *args, **kwargs):
    """Run fn once and return (result, elapsed_seconds)."""
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, time.perf_counter() - start


candidatos = {
    "Linear": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=alphas)),
    "Lasso": make_pipeline(StandardScaler(), LassoCV(alphas=alphas, max_iter=20000)),
    "kNN (k=5)": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
    "Árvore (prof. 8)": DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE),
    "SVR (rbf)": make_pipeline(StandardScaler(), SVR(kernel="rbf")),
}

linhas = []
for nome, modelo in candidatos.items():
    _, fit_s = timeit(modelo.fit, X_train, y_train)
    predicao, predict_s = timeit(modelo.predict, X_test)
    linhas.append({"modelo": nome, "rmse": root_mean_squared_error(y_test, predicao),
                   "mae": mean_absolute_error(y_test, predicao), "r2": r2_score(y_test, predicao),
                   "fit_s": fit_s, "predict_s": predict_s})

tabela = pd.DataFrame(linhas).set_index("modelo").sort_values("rmse")
display(tabela.round(4))

### Como ler isso

Olhe a coluna `r2` junto com o `rmse`: o R² é relativo à média do alvo, então ele responde "quanto melhor que chutar a média", enquanto o RMSE responde "quanto erro, na unidade do alvo". Um modelo pode ter RMSE pequeno e R² pequeno se o alvo varia pouco.

Três coisas que os números mostram e valem para outros dados:

- a **árvore** acima do grau que o dado sustenta decora. Ela é fácil de ajustar e fácil de exagerar, e é justamente por isso que os métodos de ensemble dela existem;
- o **kNN** e o **SVR** dependem de distância, então só aparecem aqui padronizados (mesma lição do notebook de classificação), e o kNN paga caro na predição;
- **nenhum modelo ganha por magia**. Aqui o SVR e o kNN, que capturam não linearidade, ficaram à frente da reta, o que diz que a relação entre estes atributos e o preço não é proporcional. Eles também são os mais caros para prever. Onde a relação for aproximadamente proporcional, o resultado se inverte e a reta ganha.

## Resumo

Regressão é escolher quanta capacidade o modelo deve ter. Pouca capacidade erra nos dois conjuntos; capacidade demais acerta no treino e erra no teste, e a curva de validação em forma de U mostra onde está o meio. Regularização entra aí: Ridge encolhe os coeficientes, Lasso zera os que não se pagam, e os dois exigem padronização para que a penalidade seja justa entre colunas.

**Próximo passo:** a mesma pergunta de capacidade, com um modelo que aprende as colunas novas em vez de recebê-las prontas, está no notebook de rede neural da pasta `deep-learning`.

### Referências

**Regularização**

Hoerl, A. E., & Kennard, R. W. (1970). *Ridge regression: biased estimation for nonorthogonal problems.* Technometrics 12(1), 55-67.

Tibshirani, R. (1996). *Regression shrinkage and selection via the lasso.* Journal of the Royal Statistical Society B 58(1), 267-288.

**O equilíbrio entre viés e variância**

Geman, S., Bienenstock, E., & Doursat, R. (1992). *Neural networks and the bias/variance dilemma.* Neural Computation 4(1), 1-58.

Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer. Capítulo 3, sobre seleção de modelo e a curva de erro de validação.

```bibtex
@article{hoerl1970ridge,
  title   = {Ridge regression: biased estimation for nonorthogonal problems},
  author  = {Hoerl, Arthur E. and Kennard, Robert W.},
  journal = {Technometrics},
  volume  = {12},
  number  = {1},
  pages   = {55--67},
  year    = {1970}
}

@article{tibshirani1996regression,
  title   = {Regression shrinkage and selection via the lasso},
  author  = {Tibshirani, Robert},
  journal = {Journal of the Royal Statistical Society: Series B},
  volume  = {58},
  number  = {1},
  pages   = {267--288},
  year    = {1996}
}

@article{geman1992neural,
  title   = {Neural networks and the bias/variance dilemma},
  author  = {Geman, Stuart and Bienenstock, Elie and Doursat, Ren{\'e}},
  journal = {Neural Computation},
  volume  = {4},
  number  = {1},
  pages   = {1--58},
  year    = {1992}
}

@book{hastie2009elements,
  title     = {The Elements of Statistical Learning},
  author    = {Hastie, Trevor and Tibshirani, Robert and Friedman, Jerome},
  year      = {2009},
  edition   = {2},
  publisher = {Springer}
}
```
